# CSE475 - Phase 2 DermNet cross-dataset model (Kaggle T4)

Trains `efficientnet_b0` on the **DermNet 23-class split** (train 14,314 / validation 1,120 / test 4,002), then
evaluates cross-dataset generalization externally.

Uses the portable harness `multi-source-skin-disease-fusion`:
- dataset downloaded from HF (`Nirob-jon/cse475-dermnet-split`, one 1.7 GB zip: `DermNetPrepared.zip`)
- checkpoint synced to HF (`Nirob-jon/cse475-skin-checkpoints`) after every epoch
- resumes from the latest HF checkpoint if this run was interrupted

**Setup required:** Add a write-scope HF token as a notebook **Secret** named `HF_TOKEN` (right panel -> Secrets -> Add secret).

In [ ]:
import os, getpass, zipfile, pathlib, tempfile, shutil
from huggingface_hub import login, snapshot_download

HF_REPO = "Nirob-jon/cse475-skin-checkpoints"
DATA_REPO = "Nirob-jon/cse475-dermnet-split"

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata  # not on Kaggle but harmless to try
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Paste your HF token (hf_...): ")
assert token, "No HF_TOKEN found"
login(token=token, add_to_git_credential=False)
print("logged in OK")

In [ ]:
# clone the harness (public repo)
!git clone --depth 1 https://github.com/nirjon001/multi-source-skin-disease-fusion.git
%cd multi-source-skin-disease-fusion

In [ ]:
# dataset: 1.7 GB zip from HF, unzipped to /kaggle/working/dermnet (contains train/validation/test)
zip_path = pathlib.Path(snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns="*.zip"))
DATA = pathlib.Path("/kaggle/working/dermnet")
shutil.rmtree(DATA, ignore_errors=True)
DATA.mkdir(parents=True)
with zipfile.ZipFile(zip_path / "DermNetPrepared.zip") as z:
    z.extractall(DATA)
print("data root:", DATA)
!ls "$DATA"
!ls "$DATA/train" | head -4

In [ ]:
# light deps only (torch/torchvision preinstalled on Kaggle)
!pip install -q timm tqdm pyyaml imagehash scikit-learn huggingface_hub

In [ ]:
# quick data sanity: class count + split sizes
import os
for split in ("train", "validation", "test"):
    cls_dirs = [d for d in os.listdir(os.path.join(str(DATA), split)) if os.path.isdir(os.path.join(str(DATA), split, d))]
    n = sum(len(os.listdir(os.path.join(str(DATA), split, c))) for c in cls_dirs)
    print(f"{split}: {len(cls_dirs)} classes, {n} images")

In [ ]:
# TRAIN (Kaggle profile: preinstalled CUDA torch, batch 32, AMP ON)
!python src/train_resumable.py --config configs/phase2_dermnet.yaml \
    --data "$DATA" --profile kaggle_t4 --hub hf --resume auto \
    --hf-repo "$HF_REPO" --epochs 15

## Results

`phase2_dermnet_last.pt` / `phase2_dermnet_best.pt` were already uploaded to HF after every epoch.
Download `results/phase2_dermnet.json` from the output panel to keep the in-domain test report.